# Step 1: vLLM 加载量化模型 — auto 识别 + quantization_config 构造 + 声明式边界

**目标**：建立 M4 声明式部署的核心认知——不止让 vLLM "读懂" 量化 config，而是理解 `quantization_config` 结构后**能自己写出来**：谁适配谁、vLLM 加载量化模型内部经历哪几步、config 字段如何驱动 vLLM 选 kernel、声明式部署的两个前提与三条件。本模块全程走 **SmoothQuant (W8A8)** 这一条主线（M2 已产出 SmoothQuant 全量化模型作为部署对象），最后能从零写出它的 compressed-tensors config。

**对应 OUTLINE 课时**：4.1 加载识别 + flag/kernel 速查 + 声明式边界（~50 分钟）。

> **单 env**：本 notebook 在 `course/m4-deploy-loop` 单 vllm env 跑。L3 的 7B 量化产物跨模块引用 M2/M3 `out/`（先跑完 M2 s5 产出 SmoothQuant 7B、M3 调优产出 tuned/final）；**0.5B 路径不需此前置**（L2 用内联 config 样本兜底）。
> **本模块算法主线**：M4 端到端只用 SmoothQuant 一种量化方案（学部署经验，非对比算法）。FP8/AWQ 的 config 样本仅作"compressed-tensors 通用结构"与"为何标准 scheme 都走 auto"的对照，不作三方法对比。

## 学完应能讲清（学完本节应能口头回答）

1. **谁适配谁**？"适配" 是 vLLM 读你的 `quantization_config` 适配你的模型，**不是你适配 vLLM**。你产出量化产物后，从 `vllm serve` 到服务起来，vLLM 替你做了什么？（提示：选架构实现类 + 选量化 kernel，你不写任何适配代码）
2. vLLM 加载量化模型内部经历哪几步？`quantization_config` 的哪个字段驱动 vLLM 给某层选哪个 kernel？（`targets`→哪些层、`scheme`→哪个 kernel、`group_size`→权重分组粒度、`ignore`→跳过哪些层）
3. compressed-tensors 的 `quantization_config` 有哪些关键字段（`config_groups`/`targets`/`scheme`/`num_bits`/`group_size`/`ignore`）？给定 **SmoothQuant W8A8** 的量化需求（"除 lm_head 外所有 Linear 层做 W8A8 对称 INT8"），你能写出对应结构吗？
4. 声明式部署的**两个前提**（架构 vLLM 已支持 + 量化用标准 scheme）+ **三条件**（标准 scheme + 权重布局符合 kernel 契约 + vLLM 补了该架构 quantized 层）；`No compatible kernel found` 对应哪个不满足？
5. 为什么 M4 全程部署 SmoothQuant (W8A8) 时**不传 `--quantization` flag** 反而对？vLLM 怎么 auto 识别它是 W8A8 INT8？（提示：读 `config.json` 的 `quantization_config` 自动识别 compressed-tensors 格式；遗留 AutoAWQ 才需 `--quantization auto_awq` 带 flag）

In [ ]:
%%capture
import pathlib, os, json
import ipytest
ipytest.autoconfig()


In [ ]:
# Setup cell（cwd 无关路径解析）。M4 跨模块读 M2/M3 的 7B 量化产物做 L3；0.5B 兜底 L2 流程。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT   = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"          # FP16 基线（s3 压测对比用）
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"         # L2/L3 轻量验证兜底
OUT_ROOT       = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
# 跨模块引用 M2/M3 的 7B 量化产物（兄弟模块 out/，只读不写）
REPO_COURSE = MODULE_ROOT.parent
M2_OUT = REPO_COURSE / "m2-quant-pipeline" / "out"      # qwen7b-fp8 / qwen7b-awq / qwen7b-smoothquant
M3_OUT = REPO_COURSE / "m3-tuning-eval" / "out"         # 调优后的 mixed-precision 产物
print("MODULE_ROOT =", MODULE_ROOT)
print("M2_OUT =", M2_OUT, "| exists:", M2_OUT.exists())
print("M3_OUT =", M3_OUT, "| exists:", M3_OUT.exists())


## 原理：声明式部署 = "你产出产物，vLLM 读 config 自动适配"

声明式部署的核心是**谁适配谁**——是 **vLLM 读你的 `quantization_config` 适配你的模型**，不是你去适配 vLLM。你只负责"产出正确的量化产物"（标准 scheme 量化 → 正确 packed 权重 + 把 `quantization_config` 写进 `config.json`），vLLM 读它自动选架构实现 + 量化 kernel——你夹在中间**不写任何适配代码**。llm-compressor 的 `save_pretrained` 自动把 `quantization_config` 写进 `config.json`，这就是"声明"的来源。

**vLLM 适配全过程（内部 6 步，`vllm serve` 启动时自动做）**：

1. 读 `config.json` 的 `architectures`（如 `Qwen2ForCausalLM`）→ 查 `ModelRegistry` 找架构实现类（attention/MLP/forward）。**层①架构适配**。
2. 读 `quantization_config`（`quant_method`）→ 查 quantization registry 找量化方法类 + 解析 `config_groups` 确定"哪些层用哪种 scheme"。**层②量化适配**。
3. 遍历模型层，给被量化的层套对应 kernel wrapper（`targets` 决定哪些层、`scheme` 决定 FP8/AWQ/INT8 哪个 kernel）。
4. 加载 packed 权重 + scale/zero_point 张量塞进 wrapper（`group_size` 决定分组粒度）。
5. **profile run**（跑 dummy 输入探测 KV-Cache 块数）——**这就是"验证"**：vLLM 启动自带、不是你写的；报错（`No compatible kernel found` / OOM）= 三条件某个不满足。
6. 起服务接请求。

**config 字段 → vLLM 适配步骤的映射**（本节构造填空的认知基础——懂了这个映射，才能从"量化需求"推导出"该写什么 config"，而非机械抄 JSON）：

| `quantization_config` 字段 | 驱动 vLLM 哪步 |
|---|---|
| `targets` | 步骤③ 哪些层套 kernel wrapper |
| `scheme`（W8A8 / W4A16 / FP8）| 步骤③ 选哪个 kernel |
| `num_bits` / `group_size` | 步骤③-④ 权重打包粒度 |
| `ignore` | 步骤③ 跳过哪些层（保持高精度）|

### Kernel 速查表：scheme 各走哪个 vLLM kernel

| 量化方法 | vLLM kernel | 是否传 flag |
|---|---|---|
| **SmoothQuant W8A8 INT8（本模块主线）** | **INT8-Marlin / scaled_mm**（Hopper `scaled_mm`、非 Hopper 回退 INT8-Marlin）| **不传（auto 识别）** |
| FP8 (W8A8) | CUTLASS `torch._scaled_mm` | 不传（auto） |
| AWQ (W4A16) | Marlin / Machete | 不传（auto） |
| 遗留 AutoAWQ | awq_marlin | **必须 `--quantization auto_awq`（带下划线）** |

> **本模块只部署 SmoothQuant**：上表只有第一行是 M4 真正用的 kernel。其余三行（FP8/AWQ/遗留 AutoAWQ）只作"compressed-tensors 通用结构 + 为何标准 scheme 都走 auto"的对照——它们的 config 构造与 vLLM 识别机制与 W8A8 同构，但 M4 不部署它们（其他量化方案见 M1/M2）。
> **为什么 `auto_awq` 带下划线？** vLLM 用 `auto_awq`（带下划线）区分**遗留 AutoAWQ 库产物**（非 compressed-tensors 格式）vs **标准 compressed-tensors AWQ**——二者是 vLLM 两个独立的 quantization registry 项。标准 compressed-tensors 产物（含 W8A8/FP8/AWQ）一律走 `auto` 无需 flag，只有遗留 AutoAWQ 格式才需显式传 `--quantization auto_awq`。这是 scheme→flag 速查里唯一"要传 flag"的特例，已在本节 ipytest 里作为已知映射断言（不再作为填空）。

### 声明式的前提：两层适配 + 三条件

声明式承诺 = "**不用写模型适配代码、不用写反量化代码**"（前提架构已通）——成立。但**不是**"随便量化都能跑"——还得"把量化做对"。

**两层适配**（vLLM 加载量化模型要两层都通）：
```
config.json
 ├─ architectures: XxxForCausalLM   →  层①架构适配（vLLM 有没有这个模型的实现类）
 └─ quantization_config: {...}       →  层②量化适配（这个 scheme vLLM 认不认）
```

**三条件**（bf16 能跑的模型，量化后直接声明式跑通需满足）：
1. **量化方案是 vLLM 认的标准 scheme**（compressed-tensors 的 FP8/AWQ/INT8、GPTQ 等，用 llm-compressor 产出）。
2. **权重布局符合 kernel 契约**（粒度/scale 形状/group_size 整除/AWQ `zero_point=False` 等）。
3. **vLLM 给该架构补了该量化的 kernel 路径**（新架构常只实现 bf16 forward、未补 quantized 层）。

**关键结论**：
- **量化不创造架构适配**：vLLM 未适配的架构，量化前后都跑不了原生路径。
- `No compatible kernel found` 通常对应条件③不满足（架构没补 quantized kernel）或条件②（权重布局不符）。

**边界外三条**（声明式不成立，仅讲不实操）：(a) 多模态变体（可能需 trust-remote-code）；(b) 非标准/自定义 quantization scheme；(c) vLLM 未原生支持的冷门架构（走 Transformers fallback `--model-impl transformers`，性能损失大，或写 model 适配 = 改代码）。


## 亲手摸一摸：真实 SmoothQuant 7B 产物的 quantization_config

看 M2/M3 产的 **SmoothQuant W8A8** 产物的 `config_groups` 字段——这是 M4 全程要部署的对象，理解"哪些层用什么 scheme"这个结构是 vLLM 步骤②③适配的依据，而不是把 config 当黑盒。FP8/AWQ 的 config 样本仅作"compressed-tensors 通用结构"对照（同样由 `config_groups`/`targets`/`weights`/`input_activations` 组成，差异只在 type/num_bits/strategy）。

In [ ]:
# 摸一摸：打印 SmoothQuant W8A8 产物的 quantization_config（M2/M3 out/，跨模块只读）
# 缺产物时用内联样本兜底（L2 独立可跑）。注意：SAMPLE_CONFIGS 是「完整 config dict」
# （含 quantization_config 键），与真实 config.json 同构——detect_quant_scheme 吃完整 config。
# FP8/AWQ 样本保留作"通用结构对照"，但 M4 只部署 W8A8。
SAMPLE_CONFIGS = {
    "W8A8 (SmoothQuant) — 本模块主线": {"architectures": ["Qwen2ForCausalLM"], "quantization_config": {
        "config_groups": {"group_0": {"format": "int-quantized",
            "targets": ["Linear"],
            "weights": {"num_bits": 8, "type": "int", "strategy": "channel", "symmetric": True},
            "input_activations": {"num_bits": 8, "type": "int", "strategy": "token", "symmetric": True, "dynamic": True}}},
        "ignore": ["lm_head"], "quant_method": "compressed-tensors"}},
    "FP8 — 结构对照（非本模块部署对象）": {"architectures": ["Qwen2ForCausalLM"], "quantization_config": {
        "config_groups": {"group_0": {"format": "float-quantized",
            "targets": ["Linear"],
            "weights": {"num_bits": 8, "type": "float", "strategy": "channel", "symmetric": True},
            "input_activations": {"num_bits": 8, "type": "float", "strategy": "token", "symmetric": True, "dynamic": True}}},
        "ignore": ["lm_head"], "quant_method": "compressed-tensors"}},
    "W4A16 (AWQ) — 结构对照（非本模块部署对象）": {"architectures": ["Qwen2ForCausalLM"], "quantization_config": {
        "config_groups": {"group_0": {"format": "pack-quantized",
            "targets": ["Linear"],
            "weights": {"num_bits": 4, "type": "int", "strategy": "group", "group_size": 128, "symmetric": False},
            "input_activations": None}},
        "ignore": ["lm_head"], "quant_method": "compressed-tensors"}},
}

# M4 部署对象：M2 的 SmoothQuant 7B + M3 调优后的 final（同 W8A8 格式）
real = {
    "M2 SmoothQuant": M2_OUT / "qwen7b-smoothquant",
    "M3 final (调优后)": M3_OUT / "s6_final",
}
print("=== W8A8 主线 + 对照方法的 config_groups 字段差异（vLLM 步骤②③的输入）===")
for name, cfg in SAMPLE_CONFIGS.items():
    qc = cfg["quantization_config"]
    g0 = qc["config_groups"]["group_0"]
    w = g0["weights"]; act = g0.get("input_activations")
    print("\n[%s] targets=%s ignore=%s" % (name, g0['targets'], qc['ignore']))
    print("  weights: type=%s num_bits=%s strategy=%s group_size=%s symmetric=%s" % (
        w['type'], w['num_bits'], w['strategy'], w.get('group_size'), w['symmetric']))
    if act is None:
        print("  input_activations: None (weight-only，激活全程 FP16)")
    else:
        print("  input_activations: type=%s dynamic=%s" % (act['type'], act.get('dynamic')))
print("\n（真 W8A8 产物路径见 L3）:", real)

## 本步填空（2 个，理解-构造递进）

1. **`detect_quant_scheme(model_path_or_config)`** — 读 `config.json` 的 `quantization_config`，返回结构化信息 dict（scheme + targets + 粒度），FP16 返回 None。**为什么这么设计（填前先想）**：先"读懂"——亲手解析 `config_groups`，理解"哪些层用什么 scheme"这个结构是 vLLM 步骤②③适配的依据，而不是把 config 当黑盒。本模块部署的 W8A8 必须被正确识别为 `'W8A8'`。
2. **`build_quantization_config(scheme, targets, num_bits=None, group_size=None, ignore=())`**（**理解型核心**）— **构造** compressed-tensors 的 `quantization_config` 字典（`config_groups` + `ignore` 结构）。**为什么这么设计**：这是"自己写出 config"——从量化需求（如 "SmoothQuant W8A8 量化除 lm_head 外所有 Linear 层"）推导出正确字段结构。**docstring 只给字段语义和约束**（`group_size` 须整除 hidden_size、`ignore` 决定哪些层保持高精度、`targets` 用 `['Linear']` 匹配），**不给逐字实参**——你要理解每个字段驱动 vLLM 哪步、该填什么值。

> **`pick_vllm_flag_and_kernel` 不再是填空**：scheme→flag/kernel 的映射在收敛到 SmoothQuant 后只剩单行映射（W8A8→`auto`、kernel=INT8-Marlin/scaled_mm），属纯查表、价值低，已**归入 ipytest 断言**（作为已知映射直接断言，而非让学员填）。下文的 `pick_vllm_flag_and_kernel` 是给 L2/L3 用的**已实现辅助函数**（非填空），其逻辑在原理 cell 的 Kernel 速查表里讲过。

In [ ]:
def detect_quant_scheme(model_path_or_config):
    """读 config.json 的 quantization_config，返回结构化信息 dict 或 None（FP16/未压缩）。
    输入：path（str/Path 指向 config.json 或模型目录），或已加载的 config dict。

    为什么这么设计（填前先想）：
    - vLLM 步骤②就是读 quantization_config 解析 config_groups——你亲手解析一遍，
      才理解 scheme 不是单一字段，而是由 weights/input_activations 的 type/num_bits/strategy 组合推断。
    - scheme 推断规则（从真实 M2 产物归纳）：
        * weights.type='float' + act.type='float'        -> 'FP8'
        * weights.type='int'  + act 存在(int)            -> 'W8A8'  (SmoothQuant INT8)
        * weights.type='int'  + act 为 None              -> 'W4A16' (AWQ weight-only)
    - FP16 / 无 quantization_config / quant_method != 'compressed-tensors' -> 返回 None。

    返回（None 或 dict，dict 含：scheme, targets, num_bits, group_size, ignore, quant_method）。
    """
    # TODO:
    #   1) 输入归一：dict 直接用；否则把 path 解析到 config.json 并 json.load。
    #   2) 取 quantization_config；为空 或 quant_method != 'compressed-tensors' -> return None。
    #   3) 取 config_groups 的 group_0（或第一个 group）。
    #   4) 按上面规则推断 scheme；从 weights 取 num_bits/group_size；从 qc 取 ignore（默认 []）。
    #   5) 返回 dict(scheme=, targets=, num_bits=, group_size=, ignore=, quant_method=)。
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# L1 测试（detect_quant_scheme）——填完 detect 立即单独跑此 cell 验证。

def test_detect_w8a8_smoothquant():
    # 本模块主线：SmoothQuant W8A8 必须被识别为 'W8A8'
    r = detect_quant_scheme(SAMPLE_CONFIGS["W8A8 (SmoothQuant) — 本模块主线"])
    assert r is not None and r["scheme"] == "W8A8"
    assert r["targets"] == ["Linear"]
    assert r["num_bits"] == 8
    assert r["ignore"] == ["lm_head"]

def test_detect_fp8_and_awq_structural():
    # 对照方法：FP8/AWQ 结构识别（非部署对象，但 detect 须能区分 scheme）
    assert detect_quant_scheme(SAMPLE_CONFIGS["FP8 — 结构对照（非本模块部署对象）"])["scheme"] == "FP8"
    r = detect_quant_scheme(SAMPLE_CONFIGS["W4A16 (AWQ) — 结构对照（非本模块部署对象）"])
    assert r["scheme"] == "W4A16" and r["group_size"] == 128 and r["num_bits"] == 4

def test_detect_fp16_returns_none():
    # FP16 基线：无 quantization_config
    assert detect_quant_scheme({"architectures": ["Qwen2ForCausalLM"]}) is None
    # quant_method 非 compressed-tensors 也 None
    assert detect_quant_scheme({"quantization_config": {"quant_method": "bnb"}}) is None

In [ ]:
def build_quantization_config(scheme, targets, num_bits=None, group_size=None, ignore=()):
    """构造 compressed-tensors 的 quantization_config 字典。

    参数（语义，不给逐字实参）：
    - scheme: 'FP8'（FP8_DYNAMIC）/ 'W4A16'（AWQ weight-only）/ 'W8A8'（SmoothQuant INT8）
    - targets: list/tuple，如 ['Linear']（驱动 vLLM 步骤③ 哪些层套 wrapper）
    - num_bits: W4A16 默认 4、W8A8/FP8 默认 8（驱动权重打包位数）
    - group_size: 仅 per-group（W4A16）有意义；W4A16 必须给（否则 ValueError）；
                  W8A8 是 per-channel，不应有 group_size（给了 >0 的值要 ValueError）
    - ignore: tuple/list，如 ('lm_head',)（驱动 vLLM 步骤③ 跳过哪些层保持高精度）

    为什么这么设计（填前先想）：
    - 每个字段都映射到 vLLM 的一步——你不是在抄 JSON，是在"指挥 vLLM 适配"。
    - 三 scheme 的权重/激活结构差异（从摸一摸 cell 真实产物归纳）：
        FP8  : weights=float/channel/sym + act=float/token/sym/dynamic；format='float-quantized'
        W4A16: weights=int/group/sym=False/group_size/zp_dtype='torch.int8' + act=None；format='pack-quantized'
        W8A8 : weights=int/channel/sym + act=int/token/sym/dynamic；format='int-quantized'
    - 输出顶层：{'config_groups': {'group_0': {...}}, 'quant_method': 'compressed-tensors',
                'format': <fmt>, 'ignore': list(ignore)}
    """
    # TODO:
    #   1) targets / ignore 先转 list。
    #   2) 按 scheme 分支构造 weights dict（含 type/strategy/symmetric/dynamic/num_bits/group_size/zp_dtype）：
    #      FP8/W8A8 还要构造 input_activations dict（token 策略、dynamic=True）；W4A16 的 act=None。
    #      W4A16 缺 group_size 报错；W8A8 给了正 group_size 报错。
    #      num_bits 缺省：FP8/W8A8->8，W4A16->4。
    #   3) group_0 = {'targets': targets, 'weights': w, 'input_activations': act,
    #                 'output_activations': None, 'format': fmt}
    #   4) 返回顶层 dict（config_groups + quant_method + format + ignore）。
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# L1 测试（build_quantization_config）——填完 build 立即单独跑此 cell 验证（不依赖 detect）。

def test_build_w8a8_smoothquant_main_line():
    # 本模块主线：构造 SmoothQuant W8A8 的 config
    qc = build_quantization_config("W8A8", ["Linear"], ignore=("lm_head",))
    assert qc["quant_method"] == "compressed-tensors"
    assert qc["ignore"] == ["lm_head"]
    g0 = qc["config_groups"]["group_0"]
    assert g0["targets"] == ["Linear"]
    assert g0["format"] == "int-quantized"
    assert g0["weights"]["type"] == "int" and g0["weights"]["num_bits"] == 8
    assert g0["weights"]["symmetric"] is True and g0["weights"]["strategy"] == "channel"
    assert g0["input_activations"]["dynamic"] is True            # 激活 dynamic per-token
    assert g0["input_activations"]["type"] == "int"

def test_build_awq_group_size():
    qc = build_quantization_config("W4A16", ["Linear"], group_size=128, ignore=["lm_head"])
    g0 = qc["config_groups"]["group_0"]
    assert g0["weights"]["num_bits"] == 4 and g0["weights"]["group_size"] == 128
    assert g0["weights"]["symmetric"] is False
    assert g0["input_activations"] is None  # weight-only
    assert g0["format"] == "pack-quantized"

def test_build_fp8_dynamic():
    qc = build_quantization_config("FP8", ["Linear"], ignore=["lm_head"])
    g0 = qc["config_groups"]["group_0"]
    assert g0["weights"]["type"] == "float" and g0["format"] == "float-quantized"
    assert g0["input_activations"]["dynamic"] is True

def test_build_awq_missing_group_size_raises():
    import pytest
    with pytest.raises(ValueError):
        build_quantization_config("W4A16", ["Linear"])

def test_build_w8a8_rejects_group_size():
    import pytest
    with pytest.raises(ValueError):
        build_quantization_config("W8A8", ["Linear"], group_size=128)

In [ ]:
# 辅助函数（非填空）：scheme -> (vllm_flag_or_None, kernel_name) 速查。
# 本模块只部署 SmoothQuant W8A8（走 auto），这个查表函数供 L2/L3 复用；
# 其映射逻辑见上方「原理：Kernel 速查表」，并在下面的 ipytest 里作为已知映射直接断言。
_SCHEME_TO_FLAG_KERNEL = {
    "FP8":      (None,       "CUTLASS scaled_mm (FP8)"),
    "W4A16":    (None,       "AWQ-Marlin / Machete (W4A16)"),
    "W8A8":     (None,       "INT8-Marlin (W8A8)"),
    "auto_awq": ("auto_awq", "awq_marlin (遗留 AutoAWQ)"),
}

def pick_vllm_flag_and_kernel(scheme):
    """给 scheme 返回 (vllm_flag_or_None, kernel_name)。未知 scheme raise KeyError。

    本模块只部署 W8A8 -> (None, INT8-Marlin)，即不传 --quantization flag（走 auto）。
    """
    if scheme not in _SCHEME_TO_FLAG_KERNEL:
        raise KeyError("未知 scheme: %r（已知: %s）" % (scheme, list(_SCHEME_TO_FLAG_KERNEL)))
    return _SCHEME_TO_FLAG_KERNEL[scheme]

In [ ]:
%%ipytest -qq
# L1 断言（pick_vllm_flag_and_kernel 的已知映射——此函数是辅助非填空，断言固化速查表）。

def test_pick_w8a8_main_line_no_flag():
    # 本模块主线：W8A8 走 auto，不传 --quantization flag
    flag, kernel = pick_vllm_flag_and_kernel("W8A8")
    assert flag is None
    assert "INT8" in kernel or "Marlin" in kernel

def test_pick_standard_schemes_all_auto():
    # 标准 compressed-tensors 产物（FP8/AWQ/W8A8）一律不传 flag
    for s in ["FP8", "W4A16", "W8A8"]:
        assert pick_vllm_flag_and_kernel(s)[0] is None

def test_pick_legacy_autoawq_needs_flag():
    # 唯一特例：遗留 AutoAWQ 必须 --quantization auto_awq（带下划线！）
    flag, kernel = pick_vllm_flag_and_kernel("auto_awq")
    assert flag == "auto_awq"   # 关键坑：带下划线，不是 'awq'
    assert "awq" in kernel

def test_pick_unknown_scheme_raises():
    import pytest
    with pytest.raises(KeyError):
        pick_vllm_flag_and_kernel("INT4")

## L2（CPU）：解析真实 config 样本验逻辑

L2 验 `detect_quant_scheme` 对真实产物/内联样本的解析正确性（CPU 可跑，不需 vllm 真加载）。主线验 M2/M3 的 **SmoothQuant W8A8** 7B 产物；缺产物用内联 `SAMPLE_CONFIGS` 兜底（保证流程层独立可跑）。

In [ ]:
## L2：解析真 W8A8 产物（M2/M3 out/）验 detect；缺产物用内联样本兜底
import pathlib
# 主线：M2 SmoothQuant + M3 调优后 final（都是 W8A8）；缺产物用内联 W8A8 样本兜底
real_dirs = {
    "M2 SmoothQuant": (M2_OUT / "qwen7b-smoothquant", "W8A8 (SmoothQuant) — 本模块主线"),
    "M3 final (调优后)": (M3_OUT / "s6_final", "W8A8 (SmoothQuant) — 本模块主线"),
}
print("=== L2：detect_quant_scheme 解析 W8A8 主线产物 ===")
saw_real = False
for label, (d, sample_key) in real_dirs.items():
    if (d / "config.json").exists():
        cfg = json.loads((d / "config.json").read_text())
        src = "真 7B 产物（%s）" % d
        saw_real = True
    else:
        cfg = SAMPLE_CONFIGS[sample_key]
        src = "内联 SAMPLE（%s 产物缺失，用兜底样本）" % d
    r = detect_quant_scheme(cfg)
    print("\n[%s] 来源=%s" % (label, src))
    print("  -> scheme={} targets={} num_bits={} group_size={} ignore={}".format(
        r["scheme"], r["targets"], r["num_bits"], r["group_size"], r["ignore"]))
    # 主线断言：W8A8 必须正确识别
    assert r["scheme"] == "W8A8", "%s 应识别为 W8A8" % label

# 验：0.5B FP16 基线必须返回 None
fp16 = json.loads((TINY_MODEL_DIR / "config.json").read_text())
assert detect_quant_scheme(fp16) is None, "FP16 基线应返回 None"
print("\nL2 通过：FP16->None、W8A8 config_groups 结构解析正确（vLLM 步骤②③依据）。")
if not saw_real:
    print("（提示：真 W8A8 7B 产物缺失，本次用内联样本验流程；先跑 M2 s5/M3 s6 产出后回看真产物。）")
print("（真 vllm LLM().generate() 验证 auto 识别 + kernel 选择见 L3）")

## L3（H200，GPU + SKIP_L3 双守卫）：0.5B 离线 generate + 7B W8A8 LLM 加载

L3 验 vLLM 真能 auto 识别 compressed-tensors 产物并选对 kernel：先在 0.5B 上离线 `vllm.LLM().generate()` 跑一句（轻量），再对 7B **SmoothQuant W8A8** 产物（M2/M3）各 `LLM()` 加载 generate。

> **L3 双守卫**：`torch.cuda.is_available() and not os.environ.get('SKIP_L3')`——reviewer 执行验证设 `SKIP_L3=1` 跳过（7B serve 分钟级，太重）；真人/学员跑时不设，L3 实证。

In [ ]:
import torch, os
def run_l3():
    # 先 0.5B 离线 LLM（轻量，验证 vllm 流程通）
    from vllm import LLM, SamplingParams
    print("[L3-0.5B] 离线 LLM().generate() 验证 vLLM 加载流程（FP16 基线）...")
    llm = LLM(model=str(TINY_MODEL_DIR), dtype="float16", enforce_eager=True, gpu_memory_utilization=0.5)
    out = llm.generate(["你好，用一句话介绍量化。"], SamplingParams(max_tokens=16, temperature=0))
    print("  0.5B 输出:", out[0].outputs[0].text.strip())
    del llm

    # 7B W8A8 主线：M2 SmoothQuant + M3 final（调优后）各 LLM() 加载验 auto 识别 + kernel 选择
    candidates = {
        "M2 SmoothQuant (W8A8)": M2_OUT / "qwen7b-smoothquant",
        "M3 final (W8A8 调优后)": M3_OUT / "s6_final",
    }
    for name, d in candidates.items():
        if not (d / "config.json").exists():
            print("\n[{}] {} 不存在——先跑 M2 s5 / M3 s6 产出 SmoothQuant W8A8 7B。".format(name, d))
            continue
        print("\n[L3-7B-{}] LLM() 加载 {}，验 auto 识别 W8A8...".format(name, d))
        llm = LLM(model=str(d), enforce_eager=True, gpu_memory_utilization=0.9)
        o = llm.generate(["2+2=?"], SamplingParams(max_tokens=8, temperature=0))
        print("  {} 输出: {} | 加载成功=vLLM auto 选对 INT8 kernel（不传 --quantization flag）".format(
            name, o[0].outputs[0].text.strip()))
        del llm

if torch.cuda.is_available() and not os.environ.get('SKIP_L3'):
    run_l3()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（reviewer 执行验证跳过真 7B vllm 跑；真人跑时不设 SKIP_L3，L3 实证）。")

## 产物检查：auto 识别成功 = 声明式闭环成立

L3 跑通即证明：vLLM 读 `config.json` 的 `quantization_config` 自动识别 compressed-tensors 格式 → 选对应 kernel → 正常 generate——**全程无需改模型代码、无需传 `--quantization` flag**（SmoothQuant W8A8 走 auto）。这就是声明式部署的闭环。

**回顾三条件**（L3 跑通 = 三条件全满足）：
1. scheme 是标准 compressed-tensors（W8A8 INT8）✅
2. 权重布局符合 kernel 契约（INT8-Marlin/scaled_mm 对称、per-channel 权重 + per-token 激活）✅
3. vLLM 给 Qwen2ForCausalLM 补了 quantized kernel 路径 ✅

若 L3 报 `No compatible kernel found`——回 §5 三条件排查（多半是条件③：架构没补 quantized kernel，或条件②：权重布局不符）。s4 会专门讲报错诊断。